In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install --only-binary=:all: "jpype1>=1.5"
!{sys.executable} -m pip -q install neo4j pandas python-dotenv 
!{sys.executable} -m pip install -q --only-binary=jpype1 pslpython
!{sys.executable} -m pip install pyvis

In [ ]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
from pyvis.network import Network
import neo4j
import pandas as pd
import os

load_dotenv()

In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()


In [ ]:
def run_cypher(query: str, params: dict | None = None) -> pd.DataFrame:
    params = params or {}
    with driver.session() as session:
        res = session.run(query, params)
        rows = [r.data() for r in res]
    return pd.DataFrame(rows)

In [ ]:
labels = run_cypher("CALL db.labels() YIELD label RETURN label ORDER BY label;")
rels = run_cypher("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType;")
labels, rels

In [ ]:
samplepersons = run_cypher("MATCH (n:Person) RETURN keys(n) AS person_props, n LIMIT 5")
samplecases = run_cypher("MATCH (n:Case) RETURN keys(n) AS case_props, n LIMIT 5")
samplelocs = run_cypher("MATCH (n:Location) RETURN keys(n) AS loc_props, n LIMIT 5")
sampleevents = run_cypher("MATCH (n:Event) RETURN keys(n) AS event_props, n LIMIT 5")
sampleentities = run_cypher("MATCH (n:Entity) RETURN keys(n) AS entity_props, n LIMIT 5")
sampleevidence = run_cypher("MATCH (n:Evidence) RETURN keys(n) AS evidence_props, n LIMIT 5")

samplepersons, samplecases, samplelocs, sampleevents, sampleentities, sampleevidence

### Visualize Database

In [ ]:
query = """
MATCH p=()-[r]->()
RETURN p
LIMIT 200
"""

net = Network(
    height="750px",
    width="100%",
    directed=True,
    notebook=False,
    cdn_resources="in_line",
)

net.repulsion(
    node_distance=300,
    central_gravity=0.1,
    spring_length=250,
    spring_strength=0.05,
    damping=0.09
)

seen_nodes = set()
seen_edges = set()

with driver.session() as session:
    result = session.run(query)

    count_paths = 0
    for record in result:
        path = record["p"]
        count_paths += 1

        for node in path.nodes:
            node_id = node.element_id
            node_label = next(iter(node.labels), "Node")
            display = node.get("name") or node.get("id") or node_label

            if node_id not in seen_nodes:
                net.add_node(
                    node_id,
                    label=display,
                    title=str(dict(node)),
                    group=node_label,
                )
                seen_nodes.add(node_id)

        for rel in path.relationships:
            edge_key = (rel.element_id,)
            if edge_key not in seen_edges:
                net.add_edge(
                    rel.start_node.element_id,
                    rel.end_node.element_id,
                    label=rel.type,
                    title=rel.type,
                )
                seen_edges.add(edge_key)

print("paths:", count_paths)
print("nodes:", len(seen_nodes))
print("edges:", len(seen_edges))

net.write_html("graph.html")

# Rule Based Layer

### Victim Related Individuals In Knowledge Graph

In [ ]:
q = """
MATCH (p:Person)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
OPTIONAL MATCH (p)--(nbr)
WITH
  c, p, r,
  count(DISTINCT nbr) AS person_degree
RETURN
  c.id AS case_id,
  c.name AS case_name,
  p.id AS person_id,
  p.name AS person_name,
  p.dob AS person_dob,
  p.type AS person_type,
  type(r) AS relationship_type,
  person_degree
LIMIT 500
"""
df_victim_individuals = run_cypher(q)
df_victim_individuals.head()

### Suspect Related Individuals In Knowledge Graph

In [ ]:
q = """
MATCH (p:Person)-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c:Case)
OPTIONAL MATCH (p)--(nbr)
WITH
  c, p, r,
  count(DISTINCT nbr) AS person_degree
RETURN
  c.id AS case_id,
  c.name AS case_name,
  p.id AS person_id,
  p.name AS person_name,
  p.dob AS person_dob,
  p.type AS person_type,
  type(r) AS relationship_type,
  person_degree
LIMIT 500
"""
df_suspects_in_cases = run_cypher(q)
df_suspects_in_cases.head()

### Hotspot Query

In [ ]:
q = """
MATCH (c:Case)<-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]-(p:Person)
MATCH (p)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
WHERE loc.name IS NOT NULL OR loc.city IS NOT NULL
WITH c, coalesce(loc.name, loc.city) AS place
WHERE place IS NOT NULL
RETURN place AS location, count(DISTINCT c) AS case_count
ORDER BY case_count DESC
LIMIT 50
"""
df_hotspots = run_cypher(q)
df_hotspots.head()

### Find Victim Related Individuals

In [ ]:
q = """
MATCH (c:Case)
OPTIONAL MATCH (p:Person)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
WITH
  c,
  count(DISTINCT p) AS persons_linked,
  collect(DISTINCT type(r)) AS relationship_types
RETURN
  c.id AS case_id,
  c.name AS case_name,
  c.status AS case_status,
  persons_linked,
  relationship_types
LIMIT 100
"""
df_case_summary = run_cypher(q)
df_case_summary.head()

### Query Individual Related Cases

In [ ]:
def query_victim_individual(id: str) -> pd.DataFrame:
    q = """
        MATCH (p:Person {id: $personId})-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        RETURN
        p.id AS personId,
        c.id AS caseId,
        collect(DISTINCT type(r)) AS relationship_types
        ORDER BY caseId
        LIMIT 50
    """
    return run_cypher(q, {"personId": id})

def query_suspect_individual(id: str) -> pd.DataFrame:
    q = """
        MATCH (p:Person {id: $personId})-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c:Case)
        RETURN
        p.id AS personId,
        c.id AS caseId,
        collect(DISTINCT type(r)) AS relationship_types
        ORDER BY caseId
        LIMIT 50
    """
    return run_cypher(q, {"personId": id})


In [ ]:
query_victim_individual("UNKNOWN_FEMALE")

In [ ]:
query_suspect_individual("UNKNOWN_FEMALE")

In [ ]:
query_victim_individual("CB")

In [ ]:
query_suspect_individual("CB")

In [88]:
query_victim_individual("KS")

,personId,caseId,relationship_types
0,KS,CASE_KS,[VICTIM_IN]


In [ ]:
query_suspect_individual("TJGE")

### Query Individual Related Locations

In [ ]:
def query_victim_individual_location(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[rc:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        OPTIONAL MATCH (p)-[rl:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        RETURN
            p.id AS personId,
            c.id AS caseId,
            type(rl) AS linkType,
            loc.id AS locationId,
            loc.name AS locationName
        ORDER BY caseId
        LIMIT 50
    """

    return run_cypher(q, {"personId": id})

def query_suspect_individual_location(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[rc:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c:Case)
        OPTIONAL MATCH (p)-[rl:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        RETURN
            p.id AS personId,
            c.id AS caseId,
            type(rc) AS personCaseRel,
            type(rl) AS linkType,
            loc.id AS locationId,
        loc.name AS locationName
        ORDER BY caseId
        LIMIT 50
    """

    return run_cypher(q, {"personId": id})

In [ ]:
query_victim_individual_location("UNKNOWN_FEMALE")

In [ ]:
query_suspect_individual_location("KI")

In [89]:
query_victim_individual_location("KS")

,personId,caseId,linkType,locationId,locationName
0,KS,CASE_KS,LAST_SEEN_NEAR,SANTA_LUCIA,Santa Lucia Hall (PF's Dorm)


In [ ]:
query_suspect_individual_location("MM")

In [ ]:
query_victim_individual_location("CB")

In [ ]:
query_suspect_individual_location("CB")

### Score Individual's Locations

In [ ]:
def score_victim_individual(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[r1:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        OPTIONAL MATCH (p)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)

        WITH p, c, loc,
            type(r1) AS personCaseRel,
            type(r2) AS caseLocRel,

            CASE type(r1)
            WHEN 'VICTIM_IN' THEN 1.0
            WHEN 'MISSING_PERSON_IN' THEN 0.9
            WHEN 'RESCUED_IN' THEN 0.85
            WHEN 'POTENTIAL_VICTIM_IN' THEN 0.6
            ELSE 0.3
            END AS s1,

            CASE type(r2)
            WHEN 'FOUND_AT' THEN 1.0
            WHEN 'LAST_SEEN_AT' THEN 0.95
            WHEN 'RESCUED_AT' THEN 0.9
            WHEN 'LAST_SEEN_NEAR' THEN 0.75
            WHEN 'PRESENT_NEAR' THEN 0.7
            WHEN 'STAYED_AT' THEN 0.65
            WHEN 'LIVED_AT' THEN 0.5
            WHEN 'LIVED_IN_OR_WORKED_NEAR' THEN 0.4
            ELSE 0.3
            END AS s2

        RETURN
        p.id AS personId,
        c.id AS caseId,
        loc.id AS locationId,
        loc.name AS locationName,
        personCaseRel,
        caseLocRel,
        s1 * s2 AS pathScore

        ORDER BY pathScore DESC, locationName
    """

    return run_cypher(q, {"personId": id})

def score_suspect_individual(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[r1:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c:Case)
        OPTIONAL MATCH (p)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)

        WITH p, c, loc,
            type(r1) AS personCaseRel,
            type(r2) AS caseLocRel,

            CASE type(r1)
            WHEN 'SUSPECT_IN' THEN 1.0
            WHEN 'ACCUSED_IN' THEN 0.9
            WHEN 'PERSON_OF_INTEREST' THEN 0.6
            ELSE 0.3
            END AS s1,

            CASE type(r2)
            WHEN 'FOUND_AT' THEN 1.0
            WHEN 'LAST_SEEN_AT' THEN 0.95
            WHEN 'RESCUED_AT' THEN 0.9
            WHEN 'LAST_SEEN_NEAR' THEN 0.75
            WHEN 'PRESENT_NEAR' THEN 0.7
            WHEN 'STAYED_AT' THEN 0.65
            WHEN 'LIVED_AT' THEN 0.5
            WHEN 'LIVED_IN_OR_WORKED_NEAR' THEN 0.4
            ELSE 0.3
            END AS s2

        RETURN
        p.id AS personId,
        c.id AS caseId,
        loc.id AS locationId,
        loc.name AS locationName,
        personCaseRel,
        caseLocRel,
        s1 * s2 AS pathScore

        ORDER BY pathScore DESC, locationName
    """

    return run_cypher(q, {"personId": id})

In [ ]:
score_victim_individual("KI")

In [ ]:
score_suspect_individual("KI")

In [ ]:
score_victim_individual("TOKYO_MAN_1")

In [ ]:
score_suspect_individual("TOKYO_MAN_1")

In [ ]:
score_victim_individual("CB")

In [ ]:
score_suspect_individual("CB")

### Find Related Individuals

In [ ]:
def find_related_victim_individuals(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        MATCH (target)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
        MATCH (other)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
        other.id AS similarPersonId,
        count(DISTINCT c) AS sharedCases,
        count(DISTINCT loc) AS sharedLocations,
        collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedCases DESC, sharedLocations DESC
        LIMIT 25
    """
    return run_cypher(q, {"personId": id})

def find_related_suspect_individuals(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c:Case)
        MATCH (target)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c)
        MATCH (other)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
        other.id AS similarPersonId,
        count(DISTINCT c) AS sharedCases,
        count(DISTINCT loc) AS sharedLocations,
        collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedCases DESC, sharedLocations DESC
        LIMIT 25
    """
    return run_cypher(q, {"personId": id})

In [ ]:
find_related_victim_individuals("TOKYO_MAN_1")

In [ ]:
find_related_suspect_individuals("TOKYO_MAN_1")

In [ ]:
find_related_victim_individuals("CB")

In [ ]:
find_related_suspect_individuals("CB")

### Find Related Individuals To Individual Location

In [ ]:
def related_individuals_to_victim_locations(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
            other.id AS similarPersonId,
            count(DISTINCT loc) AS sharedLocations,
            collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedLocations DESC
        LIMIT 25
    """

    return run_cypher(q, {"personId": id})

def related_individuals_to_suspect_locations(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(:Case)
        MATCH (target)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(:Case)
        MATCH (other)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
            other.id AS similarPersonId,
            count(DISTINCT loc) AS sharedLocations,
            collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedLocations DESC
        LIMIT 25
    """

    return run_cypher(q, {"personId": id})

In [ ]:
related_individuals_to_victim_locations("TOKYO_MAN_1")

In [ ]:
related_individuals_to_suspect_locations("MM")

In [ ]:
related_individuals_to_victim_locations("CB")

In [ ]:
related_individuals_to_suspect_locations("CB")

### Find Relations Of Individual At Location

In [ ]:
def query_victim_individual_at_location(personId : str, locationId) -> pd.DataFrame:
    q = """
        MATCH (person:Person {id: $personId})-[r1:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        MATCH (person)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location {id: $locationId})

        RETURN
            person.id AS personId,
            person.name AS personName,
            person.dob AS personDOB,
            person.type AS personType,

            c.id AS caseId,
            c.name AS caseName,
            c.status AS caseStatus,

            loc.id AS locationId,
            loc.name AS locationName,
            loc.city AS locationCity,

            type(r1) AS personCaseRel,
            type(r2) AS personLocationRel
        LIMIT 10
    """

    return run_cypher(q, {"personId": personId, "locationId": locationId})

def query_suspect_individual_at_location(personId: str, locationId: str) -> pd.DataFrame:
    q = """
        MATCH (person:Person {id: $personId})-[r1:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c:Case)
        MATCH (person)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location {id: $locationId})

        RETURN
            person.id AS personId,
            person.name AS personName,
            person.dob AS personDOB,
            person.type AS personType,

            c.id AS caseId,
            c.name AS caseName,
            c.status AS caseStatus,

            loc.id AS locationId,
            loc.name AS locationName,
            loc.city AS locationCity,

            type(r1) AS personCaseRel,
            type(r2) AS personLocationRel
        LIMIT 10
    """

    return run_cypher(q, {
        "personId": personId,
        "locationId": locationId
    })

In [ ]:
query_victim_individual_at_location("KI", "SOS_SIGN_HOLE")

In [ ]:
query_suspect_individual_at_location("KI", "SOS_SIGN_HOLE")

In [ ]:
query_victim_individual_at_location("CB", "PRAIA_DA_LUZ")

In [ ]:
query_suspect_individual_at_location("CB", "PRAIA_DA_LUZ")

# Embedding Prediction Layer

### Suspect Prediction

In [ ]:
def predict_suspect_links_common_neighbors(
    limit: int = 25,
    case_id: str | None = None
) -> pd.DataFrame:
    q = """
    MATCH (p:Person), (c:Case)

    WHERE 
        ($caseId is NULL or c.id = $caseId)
        AND NOT EXISTS {
            MATCH (p)-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

        AND NOT EXISTS {
            MATCH (p)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

    WITH p, c,
         gds.alpha.linkprediction.commonNeighbors(p, c, {direction: "BOTH"}) AS rawScore

    WHERE rawScore > 0

    OPTIONAL MATCH (p)-[fam:FAMILY_OF|FAMILY_RELATIONSHIP|PARENT_OF|SIBLING_OF|CHILD_OF|MOTHER_OF|FATHER_OF]->(:Person)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)

    WITH p, c, rawScore,
         CASE
            WHEN count(fam) > 0 THEN 0.5
            ELSE 1.0
         END AS roleWeight

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        rawScore,
        roleWeight,
        rawScore * roleWeight AS score

    ORDER BY score DESC, rawScore DESC, personId, caseId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit, "caseId": case_id})

def predict_suspect_links_adamic_adar(
    limit: int = 25,
    case_id: str | None = None
) -> pd.DataFrame:
    q = """
    MATCH (p:Person), (c:Case)

    WHERE 
        ($caseId is NULL or c.id = $caseId)
        AND NOT EXISTS {
            MATCH (p)-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

        AND NOT EXISTS {
            MATCH (p)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

    WITH p, c,
         gds.alpha.linkprediction.adamicAdar(p, c, {direction: "BOTH"}) AS rawScore

    WHERE rawScore > 0
        AND NOT EXISTS {
            MATCH (p)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
        }
    
    OPTIONAL MATCH (p)-[fam:FAMILY_OF|FAMILY_RELATIONSHIP|PARENT_OF|SIBLING_OF|CHILD_OF|MOTHER_OF|FATHER_OF]->(:Person)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)

    WITH p, c, rawScore,
         CASE
            WHEN count(fam) > 0 THEN 0.5
            ELSE 1.0
         END AS roleWeight

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        rawScore,
        roleWeight,
        rawScore * roleWeight AS score

    ORDER BY score DESC, rawScore DESC, personId, caseId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit, "caseId": case_id})

def predict_suspect_links_preferential_attachment(
    limit: int = 25,
    case_id: str | None = None
) -> pd.DataFrame:
    q = """
    MATCH (p:Person), (c:Case)

    WHERE
        ($caseId is NULL or c.id = $caseId)
        AND NOT EXISTS {
            MATCH (p)-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

        AND NOT EXISTS {
            MATCH (p)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

    WITH p, c,
         gds.alpha.linkprediction.preferentialAttachment(p, c) AS rawScore

    WHERE rawScore > 0
        AND NOT EXISTS {
            MATCH (p)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
        }
    OPTIONAL MATCH (p)-[fam:FAMILY_OF|FAMILY_RELATIONSHIP|PARENT_OF|SIBLING_OF|CHILD_OF|MOTHER_OF|FATHER_OF]->(:Person)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)

    WITH p, c, rawScore,
         CASE
            WHEN count(fam) > 0 THEN 0.5
            ELSE 1.0
         END AS roleWeight

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        rawScore,
        roleWeight,
        rawScore * roleWeight AS score

    ORDER BY score DESC, rawScore DESC, personId, caseId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit, "caseId": case_id})

In [90]:
predict_suspect_links_common_neighbors(case_id="CASE_KS")

757927 [neo4j.notifications PSL] WARNING --- Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SIBLING_OF` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=21, column=69, offset=641>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 641, 'line': 21, 'column': 69}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (p:Person), (c:Case)\n\n    WHERE \n        ($caseId is NULL or c.id = $caseId)\n        AND NOT EXISTS {\n            MATCH (p)-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)\n            WHERE coalesce(r.hidden_for_test, false) = false\n        }\n\n  

,personId,personName,caseId,caseName,rawScore,roleWeight,score
0,CA,Cheryl Anderson,CASE_KS,Murder of Kristin Smart,2.0,1.0,2.0
1,TD,Tim Davis,CASE_KS,Murder of Kristin Smart,2.0,1.0,2.0
2,DS,Denise Smart,CASE_KS,Murder of Kristin Smart,2.0,0.5,1.0
3,SS,Stan Smart,CASE_KS,Murder of Kristin Smart,2.0,0.5,1.0


In [ ]:
predict_suspect_links_adamic_adar(case_id="CASE_KS")

In [ ]:
predict_suspect_links_preferential_attachment(case_id="CASE_KS")

### Location Prediction

In [ ]:
def predict_case_location_common_neighbors(
    limit: int = 25,
    case_id: str | None = None
) -> pd.DataFrame:
    q = """
    MATCH (c:Case), (l:Location)

    WHERE
        ($caseId IS NULL OR c.id = $caseId)

        AND NOT EXISTS {
            MATCH (c)-[r:HAS_LOCATION|FOUND_AT|SEARCHED_AT|INVESTIGATED_IN|RESCUED_AT]->(l)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

    WITH c, l,
         gds.alpha.linkprediction.commonNeighbors(c, l, {direction: "BOTH"}) AS rawScore

    WHERE rawScore > 0

    OPTIONAL MATCH (c)<-[
        :MISSING_PERSON_IN|
        :POTENTIAL_VICTIM_IN|
        :RESCUED_IN|
        :VICTIM_IN
    ]-(:Person)-[strongLocRel:FOUND_AT|LAST_SEEN_AT|RESCUED_AT]->(l)

    WITH c, l, rawScore,
         CASE
            WHEN count(strongLocRel) > 0 THEN 1.5
            ELSE 1.0
         END AS roleWeight

    RETURN
        c.id AS caseId,
        c.name AS caseName,
        l.id AS locationId,
        l.name AS locationName,
        l.city AS locationCity,
        rawScore,
        roleWeight,
        rawScore * roleWeight AS score

    ORDER BY score DESC, rawScore DESC, caseId, locationId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit, "caseId": case_id})

def predict_case_location_adamic_adar(
    limit: int = 25,
    case_id: str | None = None
) -> pd.DataFrame:
    q = """
    MATCH (c:Case), (l:Location)

    WHERE
        ($caseId IS NULL OR c.id = $caseId)

        AND NOT EXISTS {
            MATCH (c)-[r:HAS_LOCATION|FOUND_AT|SEARCHED_AT|INVESTIGATED_IN|RESCUED_AT]->(l)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

    WITH c, l,
         gds.alpha.linkprediction.adamicAdar(c, l, {direction: "BOTH"}) AS rawScore

    WHERE rawScore > 0

    OPTIONAL MATCH (c)<-[
        :MISSING_PERSON_IN|
        :POTENTIAL_VICTIM_IN|
        :RESCUED_IN|
        :VICTIM_IN
    ]-(:Person)-[strongLocRel:FOUND_AT|LAST_SEEN_AT|RESCUED_AT]->(l)

    WITH c, l, rawScore,
         CASE
            WHEN count(strongLocRel) > 0 THEN 1.5
            ELSE 1.0
         END AS roleWeight

    RETURN
        c.id AS caseId,
        c.name AS caseName,
        l.id AS locationId,
        l.name AS locationName,
        l.city AS locationCity,
        rawScore,
        roleWeight,
        rawScore * roleWeight AS score

    ORDER BY score DESC, rawScore DESC, caseId, locationId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit, "caseId": case_id})


def predict_case_location_preferential_attachment(
    limit: int = 25,
    case_id: str | None = None
) -> pd.DataFrame:
    q = """
    MATCH (c:Case), (l:Location)

    WHERE
        ($caseId IS NULL OR c.id = $caseId)

        AND NOT EXISTS {
            MATCH (c)-[r:HAS_LOCATION|FOUND_AT|SEARCHED_AT|INVESTIGATED_IN|RESCUED_AT]->(l)
            WHERE coalesce(r.hidden_for_test, false) = false
        }

    WITH c, l,
         gds.alpha.linkprediction.preferentialAttachment(c, l) AS rawScore

    WHERE rawScore > 0

    OPTIONAL MATCH (c)<-[
        :MISSING_PERSON_IN|
        :POTENTIAL_VICTIM_IN|
        :RESCUED_IN|
        :VICTIM_IN
    ]-(:Person)-[strongLocRel:FOUND_AT|LAST_SEEN_AT|RESCUED_AT]->(l)

    WITH c, l, rawScore,
         CASE
            WHEN count(strongLocRel) > 0 THEN 1.5
            ELSE 1.0
         END AS roleWeight

    RETURN
        c.id AS caseId,
        c.name AS caseName,
        l.id AS locationId,
        l.name AS locationName,
        l.city AS locationCity,
        rawScore,
        roleWeight,
        rawScore * roleWeight AS score

    ORDER BY score DESC, rawScore DESC, caseId, locationId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit, "caseId": case_id})

In [91]:
predict_case_location_common_neighbors(case_id="CASE_KS")

774791 [neo4j.notifications PSL] WARNING --- Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. ':MISSING_PERSON_IN|:POTENTIAL_VICTIM_IN|:RESCUED_IN|:VICTIM_IN' is deprecated. It is replaced by ':MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN'.", position=<SummaryInputPosition line=20, column=20, offset=511>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 511, 'line': 20, 'column': 20}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (c:Case), (l:Location)\n\n    WHERE\n        ($caseId IS NULL OR c.id = $caseId)\n\n        AND NOT EXISTS {\n            MATCH (c)-[r:HAS_LOCATION|FOUND_AT|SEARCHED_AT|INVESTIGATED_IN|

,caseId,caseName,locationId,locationName,locationCity,rawScore,roleWeight,score
0,CASE_KS,Murder of Kristin Smart,PARTY_LOC,Crandall Way Party House,San Luis Obispo,2.0,1.0,2.0
1,CASE_KS,Murder of Kristin Smart,SANTA_LUCIA,Santa Lucia Hall (PF's Dorm),San Luis Obispo,2.0,1.0,2.0
2,CASE_KS,Murder of Kristin Smart,PF_HOME_LA,Paul Flores Home,San Pedro,1.0,1.0,1.0
3,CASE_KS,Murder of Kristin Smart,RF_HOME,Ruben Flores Home,Arroyo Grande,1.0,1.0,1.0


In [ ]:
predict_case_location_adamic_adar(case_id="CASE_KS")

In [ ]:
predict_case_location_preferential_attachment(case_id="CASE_KS")

In [ ]:
def inspect_person_case_shared_neighbors(person_id=None, case_id=None):
    q = """
    MATCH (p:Person)-[r1]-(x)-[r2]-(c:Case)

    WHERE ($personId IS NULL OR p.id = $personId)
      AND ($caseId IS NULL OR c.id = $caseId)

    RETURN
        p.id AS personId,
        c.id AS caseId,
        labels(x) AS sharedNeighborLabels,
        x.id AS sharedNeighborId,
        type(r1) AS personRelation,
        type(r2) AS caseRelation

    ORDER BY personId, caseId
    LIMIT 100
    """

    return run_cypher(q, {
        "personId": person_id,
        "caseId": case_id
    })

inspect_person_case_shared_neighbors()

## PSL Layer

In [ ]:
from pslpython.model import Model
from pslpython.predicate import Predicate
from pslpython.partition import Partition

In [ ]:
# Aggregate Relationship Types Into Larger Ones
AGG_PERSON_IN_CASE = [
    "VICTIM_IN",
    "POTENTIAL_VICTIM_IN",
    "MISSING_PERSON_IN",
    "SUSPECT_IN",
    "PERSON_OF_INTEREST",
    "ACCUSED_IN",
    "FAMILY_INVOLVED_IN",
]

AGG_PERSON_AT_LOCATION = [
    "LAST_SEEN_AT",
    "LAST_SEEN_NEAR",
    "LIVED_AT",
    "LIVED_IN_OR_WORKED_NEAR",
    "PRESENT_NEAR",
    "RESCUED_IN",
    "STAYED_AT",
    "STATIONED_AT",
    "RESIDENCE_OF",
    "ACCOMPANIED_ON_NIGHT",
]

AGG_CASE_AT_LOCATION = [
    "FOUND_AT",
    "RESCUED_AT",
    "SEARCHED_AT",
    "INVESTIGATED_IN",
    "HAS_LOCATION",
    "ATTENDED_PARTY_AT",
    "ANALYZED_AT",
    "FILED_CIVIL_SUIT_IN",
]

AGG_RELATED_PERSON = [
    "ACCOMPANIED_BY",
    "FAMILY_OF",
    "FAMILY_RELATIONSHIP",
]

AGG_EVIDENCE_LOCATION = [
    "EVIDENCE_SEIZED",
    "SEIZED_FROM",
    "SEIZED_IN",
]

AGG_VICTIM_IN_CASE = [
    "VICTIM_IN",
    "POTENTIAL_VICTIM_IN",
    "MISSING_PERSON_IN",
    "RESCUED_IN",
]

AGG_SUSPECT_IN_CASE = [
    "SUSPECT_IN",
    "PERSON_OF_INTEREST",
    "ACCUSED_IN",
]

AGG_FAMILY_IN_CASE = [
    "FAMILY_INVOLVED_IN",
]

AGG_EVIDENCE_AT_LOCATION = [
    "EVIDENCE_SEIZED",
    "SEIZED_IN",
    "SEIZED_FROM",
    "SEARCHED_AT",
    "FOUND_AT",
    "ANALYZED_AT",
]

In [ ]:
def add_predicate_helper(model, predicate_name, closed=True, size=2) -> None:
    model.add_predicate(Predicate(predicate_name, closed=closed, size=size, ))

def export_binary_relation(rel_type, start_label, end_label, filename):
    q = f"""
    MATCH (a:{start_label})-[r:{rel_type}]->(b:{end_label})
    RETURN DISTINCT
        coalesce(a.id, a.name, toString(id(a))) AS a,
        coalesce(b.id, b.name, toString(id(b))) AS b
    """
    df = run_cypher(q)

    df = df.dropna()
    df = df.astype(str)
    df = df.drop_duplicates()

    df.to_csv(filename, sep="\t", header=False, index=False)
    return df

def export_aggregated_relation(rel_types, start_label, end_label, filename):
    q = f"""
    MATCH (a:{start_label})-[r]->(b:{end_label})
    WHERE type(r) IN $rel_types
    RETURN DISTINCT
        coalesce(a.id, a.name, toString(id(a))) AS a,
        coalesce(b.id, b.name, toString(id(b))) AS b
    """

    df = run_cypher(q, {"rel_types": rel_types})
    df = df.dropna().astype(str).drop_duplicates()
    df.to_csv(filename, sep="\t", header=False, index=False)

    print(f"{filename}: {len(df)} rows")
    return df

def add_data_if_nonempty(model, predicate_name, partition, filename, df):
    if df is None or df.empty:
        print(f"Skipping {predicate_name}: {filename} is empty")
        return

    model.get_predicate(predicate_name).add_data_file(
        partition,
        filename
    )

def build_psl_model():
    model = Model("missing_person")

    # Predicates
    add_predicate_helper(model, "EvidenceAtLocation")
    add_predicate_helper(model, "PersonLinkedToEvidence")

    add_predicate_helper(model, "PersonInCase")
    add_predicate_helper(model, "PersonAtLocation")
    add_predicate_helper(model, "CaseAtLocation")
    add_predicate_helper(model, "RelatedPerson")

    add_predicate_helper(model, "LikelyLocationInCase", size=3, closed=False)

    add_predicate_helper(model, "VictimInCase")
    add_predicate_helper(model, "SuspectInCase")
    add_predicate_helper(model, "FamilyInCase")

    add_predicate_helper(model, "LikelySuspectInCase", size=2, closed=False)

    # Rules
    model.add_rule("1.0: PersonInCase(P, C) & CaseAtLocation(C, L) -> LikelyLocationInCase(P, C, L) ^2")
    model.add_rule("4.0: PersonInCase(P, C) & PersonAtLocation(P, L) -> LikelyLocationInCase(P, C, L) ^2")
    model.add_rule("0.5: PersonInCase(P1, C) & PersonInCase(P2, C) & RelatedPerson(P1, P2) & PersonAtLocation(P2, L) -> LikelyLocationInCase(P1, C, L) ^2")

    model.add_rule("3.0: PersonAtLocation(P, L) & CaseAtLocation(C, L) & !VictimInCase(P, C) -> LikelySuspectInCase(P, C) ^2")
    model.add_rule("2.0: RelatedPerson(P, V) & VictimInCase(V, C) & !FamilyInCase(P, C) -> LikelySuspectInCase(P, C) ^2")
    model.add_rule("1.0: PersonInCase(P, C) & PersonAtLocation(P, L) & CaseAtLocation(C, L) & !VictimInCase(P, C) -> LikelySuspectInCase(P, C) ^2")

    model.add_rule("5.0: VictimInCase(P, C) -> !LikelySuspectInCase(P, C) ^2")
    model.add_rule("2.0: FamilyInCase(P, C) -> !LikelySuspectInCase(P, C) ^2")

    model.add_rule("4.0: PersonLinkedToEvidence(P, E) & EvidenceAtLocation(E, L) & CaseAtLocation(C, L) & !VictimInCase(P, C) -> LikelySuspectInCase(P, C) ^2")
    model.add_rule("3.0: PersonAtLocation(P, L) & EvidenceAtLocation(E, L) & CaseAtLocation(C, L) & !VictimInCase(P, C) -> LikelySuspectInCase(P, C) ^2")
    model.add_rule("5.0: SuspectInCase(P, C) -> LikelySuspectInCase(P, C) ^2")
    return model

def load_model(model):
    evidence_at_location_df = export_aggregated_relation(
        AGG_EVIDENCE_LOCATION,
        "Evidence",
        "Location",
        "evidence_at_location.tsv"
    )

    person_linked_to_evidence_df = export_aggregated_relation(
        [
            "OWNED_BY",
            "BELONGS_TO",
            "LINKED_TO",
            "CONNECTED_TO",
            "FOUND_WITH",
            "ASSOCIATED_WITH"
        ],
        "Person",
        "Evidence",
        "person_linked_to_evidence.tsv"
    )

    person_in_case_df = export_aggregated_relation(
        AGG_PERSON_IN_CASE,
        "Person",
        "Case",
        "person_in_case.tsv"
    )

    person_at_location_df = export_aggregated_relation(
        AGG_PERSON_AT_LOCATION,
        "Person",
        "Location",
        "person_at_location.tsv"
    )

    case_at_location_df = export_aggregated_relation(
        AGG_CASE_AT_LOCATION,
        "Case",
        "Location",
        "case_at_location.tsv"
    )

    related_person_df = export_aggregated_relation(
        AGG_RELATED_PERSON,
        "Person",
        "Person",
        "related_person.tsv"
    )

    victim_in_case_df = export_aggregated_relation(
        AGG_VICTIM_IN_CASE,
        "Person",
        "Case",
        "victim_in_case.tsv"
    )

    suspect_in_case_df = export_aggregated_relation(
        AGG_SUSPECT_IN_CASE,
        "Person",
        "Case",
        "suspect_in_case.tsv"
    )

    family_in_case_df = export_aggregated_relation(
        AGG_FAMILY_IN_CASE,
        "Person",
        "Case",
        "family_in_case.tsv"
    )

    add_data_if_nonempty(
        model, "PersonInCase", 
        Partition.OBSERVATIONS,
        "person_in_case.tsv", 
        person_in_case_df
    )

    add_data_if_nonempty(
        model, "PersonAtLocation", 
        Partition.OBSERVATIONS,
        "person_at_location.tsv", 
        person_at_location_df
    )

    add_data_if_nonempty(
        model, "CaseAtLocation", 
        Partition.OBSERVATIONS,
        "case_at_location.tsv", 
        case_at_location_df
    )

    add_data_if_nonempty(
        model, "RelatedPerson", 
        Partition.OBSERVATIONS,
        "related_person.tsv", 
        related_person_df
    )

    add_data_if_nonempty(
        model,
        "VictimInCase",
        Partition.OBSERVATIONS,
        "victim_in_case.tsv",
        victim_in_case_df
    )

    add_data_if_nonempty(
        model,
        "SuspectInCase",
        Partition.OBSERVATIONS,
        "suspect_in_case.tsv",
        suspect_in_case_df
    )

    add_data_if_nonempty(
        model,
        "FamilyInCase",
        Partition.OBSERVATIONS,
        "family_in_case.tsv",
        family_in_case_df
    )

    add_data_if_nonempty(
        model,
        "EvidenceAtLocation",
        Partition.OBSERVATIONS,
        "evidence_at_location.tsv",
        evidence_at_location_df
    )

    add_data_if_nonempty(
        model,
        "PersonLinkedToEvidence",
        Partition.OBSERVATIONS,
        "person_linked_to_evidence.tsv",
        person_linked_to_evidence_df
    )

def fill_prediate_into_model(model):
    targets_from_case_locations = run_cypher("""
            MATCH (p:Person)-[r1]->(c:Case)
            MATCH (c)-[r2]->(l:Location)
            WHERE type(r1) IN $person_in_case_rels
            AND type(r2) IN $case_at_location_rels
            RETURN DISTINCT
                coalesce(p.id, p.name, toString(id(p))) AS person,
                coalesce(c.id, c.name, toString(id(c))) AS case_id,
                coalesce(l.id, l.name, toString(id(l))) AS location
        """, {
            "person_in_case_rels": AGG_PERSON_IN_CASE,
            "case_at_location_rels": AGG_CASE_AT_LOCATION,
        })

    targets_from_person_locations = run_cypher("""
            MATCH (p:Person)-[r1]->(c:Case)
            MATCH (p)-[r2]->(l:Location)
            WHERE type(r1) IN $person_in_case_rels
            AND type(r2) IN $person_at_location_rels
            RETURN DISTINCT
                coalesce(p.id, p.name, toString(id(p))) AS person,
                coalesce(c.id, c.name, toString(id(c))) AS case_id,
                coalesce(l.id, l.name, toString(id(l))) AS location
        """, {
            "person_in_case_rels": AGG_PERSON_IN_CASE,
            "person_at_location_rels": AGG_PERSON_AT_LOCATION,
        })

    targets_from_related_person_locations = run_cypher("""
            MATCH (p1:Person)-[r1]->(c:Case)
            MATCH (p2:Person)-[r2]->(c)
            MATCH (p1)-[r3]-(p2)
            MATCH (p2)-[r4]->(l:Location)
            WHERE type(r1) IN $person_in_case_rels
            AND type(r2) IN $person_in_case_rels
            AND type(r3) IN $related_person_rels
            AND type(r4) IN $person_at_location_rels
            RETURN DISTINCT
                coalesce(p1.id, p1.name, toString(id(p1))) AS person,
                coalesce(c.id, c.name, toString(id(c))) AS case_id,
                coalesce(l.id, l.name, toString(id(l))) AS location
        """, {
            "person_in_case_rels": AGG_PERSON_IN_CASE,
            "related_person_rels": AGG_RELATED_PERSON,
            "person_at_location_rels": AGG_PERSON_AT_LOCATION,
        })

    likely_location_targets = pd.concat(
        [
            targets_from_case_locations,
            targets_from_person_locations,
            targets_from_related_person_locations,
        ],
        ignore_index=True
    )

    likely_location_targets = (
        likely_location_targets
        .dropna()
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates(subset=["person", "case_id", "location"])
    )

    likely_location_targets.to_csv(
        "likely_location_targets.tsv",
        sep="\t",
        header=False,
        index=False
    )

    model.get_predicate("LikelyLocationInCase").add_data_file(
        Partition.TARGETS,
        "likely_location_targets.tsv"
    )

    targets_from_shared_case_locations = run_cypher("""
        MATCH (p:Person)-[r1]->(l:Location)
        MATCH (c:Case)-[r2]->(l)
        WHERE type(r1) IN $person_at_location_rels
        AND type(r2) IN $case_at_location_rels

        AND NOT EXISTS {
            MATCH (p)-[v]->(c)
            WHERE type(v) IN $victim_in_case_rels
        }

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "person_at_location_rels": AGG_PERSON_AT_LOCATION,
        "case_at_location_rels": AGG_CASE_AT_LOCATION,
        "victim_in_case_rels": AGG_VICTIM_IN_CASE,
    })

    targets_from_existing_suspect_roles = run_cypher("""
        MATCH (p:Person)-[r]->(c:Case)
        WHERE type(r) IN $suspect_in_case_rels

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "suspect_in_case_rels": AGG_SUSPECT_IN_CASE,
    })

    targets_from_related_to_victims = run_cypher("""
        MATCH (p:Person)-[r1]-(v:Person)-[r2]->(c:Case)
        WHERE type(r1) IN $related_person_rels
        AND type(r2) IN $victim_in_case_rels

        AND NOT EXISTS {
            MATCH (p)-[vrel]->(c)
            WHERE type(vrel) IN $victim_in_case_rels
        }

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "related_person_rels": AGG_RELATED_PERSON,
        "victim_in_case_rels": AGG_VICTIM_IN_CASE,
    })

    targets_from_person_case_context = run_cypher("""
        MATCH (p:Person)-[r]->(c:Case)
        WHERE type(r) IN $person_in_case_rels
        AND NOT type(r) IN $victim_in_case_rels

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "person_in_case_rels": AGG_PERSON_IN_CASE,
        "victim_in_case_rels": AGG_VICTIM_IN_CASE,
    })

    likely_suspect_targets = pd.concat(
        [
            targets_from_shared_case_locations,
            targets_from_existing_suspect_roles,
            targets_from_related_to_victims,
            targets_from_person_case_context,
        ],
        ignore_index=True
    )

    likely_suspect_targets = (
        likely_suspect_targets
        .dropna()
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates(subset=["person", "case_id"])
    )

    likely_suspect_targets.to_csv(
        "likely_suspect_targets.tsv",
        sep="\t",
        header=False,
        index=False
    )

    model.get_predicate("LikelySuspectInCase").add_data_file(
        Partition.TARGETS,
        "likely_suspect_targets.tsv"
    )

def run_psl():
    model = build_psl_model()
    load_model(model)
    fill_prediate_into_model(model)
    return model.infer()

In [ ]:
def export_binary_relation(rel_type, start_label, end_label, filename):
    q = f"""
    MATCH (a:{start_label})-[r:{rel_type}]->(b:{end_label})
    RETURN DISTINCT
        coalesce(a.id, a.name, toString(id(a))) AS a,
        coalesce(b.id, b.name, toString(id(b))) AS b
    """
    df = run_cypher(q)

    df = df.dropna()
    df = df.astype(str)
    df = df.drop_duplicates()

    df.to_csv(filename, sep="\t", header=False, index=False)
    return df

def export_aggregated_relation(rel_types, start_label, end_label, filename):
    q = f"""
    MATCH (a:{start_label})-[r]->(b:{end_label})
    WHERE type(r) IN $rel_types
    RETURN DISTINCT
        coalesce(a.id, a.name, toString(id(a))) AS a,
        coalesce(b.id, b.name, toString(id(b))) AS b
    """

    df = run_cypher(q, {"rel_types": rel_types})
    df = df.dropna().astype(str).drop_duplicates()
    df.to_csv(filename, sep="\t", header=False, index=False)

    print(f"{filename}: {len(df)} rows")
    return df

def add_data_if_nonempty(model, predicate_name, partition, filename, df):
    if df is None or df.empty:
        print(f"Skipping {predicate_name}: {filename} is empty")
        return

    model.get_predicate(predicate_name).add_data_file(
        partition,
        filename
    )

def load_model(model):
    evidence_at_location_df = export_aggregated_relation(
        AGG_EVIDENCE_LOCATION,
        "Evidence",
        "Location",
        "evidence_at_location.tsv"
    )

    person_linked_to_evidence_df = export_aggregated_relation(
        [
            "OWNED_BY",
            "BELONGS_TO",
            "LINKED_TO",
            "CONNECTED_TO",
            "FOUND_WITH",
            "ASSOCIATED_WITH"
        ],
        "Person",
        "Evidence",
        "person_linked_to_evidence.tsv"
    )

    person_in_case_df = export_aggregated_relation(
        AGG_PERSON_IN_CASE,
        "Person",
        "Case",
        "person_in_case.tsv"
    )

    person_at_location_df = export_aggregated_relation(
        AGG_PERSON_AT_LOCATION,
        "Person",
        "Location",
        "person_at_location.tsv"
    )

    case_at_location_df = export_aggregated_relation(
        AGG_CASE_AT_LOCATION,
        "Case",
        "Location",
        "case_at_location.tsv"
    )

    related_person_df = export_aggregated_relation(
        AGG_RELATED_PERSON,
        "Person",
        "Person",
        "related_person.tsv"
    )

    victim_in_case_df = export_aggregated_relation(
        AGG_VICTIM_IN_CASE,
        "Person",
        "Case",
        "victim_in_case.tsv"
    )

    suspect_in_case_df = export_aggregated_relation(
        AGG_SUSPECT_IN_CASE,
        "Person",
        "Case",
        "suspect_in_case.tsv"
    )

    family_in_case_df = export_aggregated_relation(
        AGG_FAMILY_IN_CASE,
        "Person",
        "Case",
        "family_in_case.tsv"
    )

    add_data_if_nonempty(
        model, "PersonInCase", 
        Partition.OBSERVATIONS,
        "person_in_case.tsv", 
        person_in_case_df
    )

    add_data_if_nonempty(
        model, "PersonAtLocation", 
        Partition.OBSERVATIONS,
        "person_at_location.tsv", 
        person_at_location_df
    )

    add_data_if_nonempty(
        model, "CaseAtLocation", 
        Partition.OBSERVATIONS,
        "case_at_location.tsv", 
        case_at_location_df
    )

    add_data_if_nonempty(
        model, "RelatedPerson", 
        Partition.OBSERVATIONS,
        "related_person.tsv", 
        related_person_df
    )

    add_data_if_nonempty(
        model,
        "VictimInCase",
        Partition.OBSERVATIONS,
        "victim_in_case.tsv",
        victim_in_case_df
    )

    add_data_if_nonempty(
        model,
        "SuspectInCase",
        Partition.OBSERVATIONS,
        "suspect_in_case.tsv",
        suspect_in_case_df
    )

    add_data_if_nonempty(
        model,
        "FamilyInCase",
        Partition.OBSERVATIONS,
        "family_in_case.tsv",
        family_in_case_df
    )

    add_data_if_nonempty(
        model,
        "EvidenceAtLocation",
        Partition.OBSERVATIONS,
        "evidence_at_location.tsv",
        evidence_at_location_df
    )

    add_data_if_nonempty(
        model,
        "PersonLinkedToEvidence",
        Partition.OBSERVATIONS,
        "person_linked_to_evidence.tsv",
        person_linked_to_evidence_df
    )

def fill_prediate_into_model(model):
    targets_from_case_locations = run_cypher("""
            MATCH (p:Person)-[r1]->(c:Case)
            MATCH (c)-[r2]->(l:Location)
            WHERE type(r1) IN $person_in_case_rels
            AND type(r2) IN $case_at_location_rels
            RETURN DISTINCT
                coalesce(p.id, p.name, toString(id(p))) AS person,
                coalesce(c.id, c.name, toString(id(c))) AS case_id,
                coalesce(l.id, l.name, toString(id(l))) AS location
        """, {
            "person_in_case_rels": AGG_PERSON_IN_CASE,
            "case_at_location_rels": AGG_CASE_AT_LOCATION,
        })

    targets_from_person_locations = run_cypher("""
            MATCH (p:Person)-[r1]->(c:Case)
            MATCH (p)-[r2]->(l:Location)
            WHERE type(r1) IN $person_in_case_rels
            AND type(r2) IN $person_at_location_rels
            RETURN DISTINCT
                coalesce(p.id, p.name, toString(id(p))) AS person,
                coalesce(c.id, c.name, toString(id(c))) AS case_id,
                coalesce(l.id, l.name, toString(id(l))) AS location
        """, {
            "person_in_case_rels": AGG_PERSON_IN_CASE,
            "person_at_location_rels": AGG_PERSON_AT_LOCATION,
        })

    targets_from_related_person_locations = run_cypher("""
            MATCH (p1:Person)-[r1]->(c:Case)
            MATCH (p2:Person)-[r2]->(c)
            MATCH (p1)-[r3]-(p2)
            MATCH (p2)-[r4]->(l:Location)
            WHERE type(r1) IN $person_in_case_rels
            AND type(r2) IN $person_in_case_rels
            AND type(r3) IN $related_person_rels
            AND type(r4) IN $person_at_location_rels
            RETURN DISTINCT
                coalesce(p1.id, p1.name, toString(id(p1))) AS person,
                coalesce(c.id, c.name, toString(id(c))) AS case_id,
                coalesce(l.id, l.name, toString(id(l))) AS location
        """, {
            "person_in_case_rels": AGG_PERSON_IN_CASE,
            "related_person_rels": AGG_RELATED_PERSON,
            "person_at_location_rels": AGG_PERSON_AT_LOCATION,
        })

    likely_location_targets = pd.concat(
        [
            targets_from_case_locations,
            targets_from_person_locations,
            targets_from_related_person_locations,
        ],
        ignore_index=True
    )

    likely_location_targets = (
        likely_location_targets
        .dropna()
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates(subset=["person", "case_id", "location"])
    )

    likely_location_targets.to_csv(
        "likely_location_targets.tsv",
        sep="\t",
        header=False,
        index=False
    )

    model.get_predicate("LikelyLocationInCase").add_data_file(
        Partition.TARGETS,
        "likely_location_targets.tsv"
    )

    targets_from_shared_case_locations = run_cypher("""
        MATCH (p:Person)-[r1]->(l:Location)
        MATCH (c:Case)-[r2]->(l)
        WHERE type(r1) IN $person_at_location_rels
        AND type(r2) IN $case_at_location_rels

        AND NOT EXISTS {
            MATCH (p)-[v]->(c)
            WHERE type(v) IN $victim_in_case_rels
        }

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "person_at_location_rels": AGG_PERSON_AT_LOCATION,
        "case_at_location_rels": AGG_CASE_AT_LOCATION,
        "victim_in_case_rels": AGG_VICTIM_IN_CASE,
    })

    targets_from_existing_suspect_roles = run_cypher("""
        MATCH (p:Person)-[r]->(c:Case)
        WHERE type(r) IN $suspect_in_case_rels

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "suspect_in_case_rels": AGG_SUSPECT_IN_CASE,
    })

    targets_from_related_to_victims = run_cypher("""
        MATCH (p:Person)-[r1]-(v:Person)-[r2]->(c:Case)
        WHERE type(r1) IN $related_person_rels
        AND type(r2) IN $victim_in_case_rels

        AND NOT EXISTS {
            MATCH (p)-[vrel]->(c)
            WHERE type(vrel) IN $victim_in_case_rels
        }

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "related_person_rels": AGG_RELATED_PERSON,
        "victim_in_case_rels": AGG_VICTIM_IN_CASE,
    })

    targets_from_person_case_context = run_cypher("""
        MATCH (p:Person)-[r]->(c:Case)
        WHERE type(r) IN $person_in_case_rels
        AND NOT type(r) IN $victim_in_case_rels

        RETURN DISTINCT
            coalesce(p.id, p.name, toString(id(p))) AS person,
            coalesce(c.id, c.name, toString(id(c))) AS case_id
    """, {
        "person_in_case_rels": AGG_PERSON_IN_CASE,
        "victim_in_case_rels": AGG_VICTIM_IN_CASE,
    })

    likely_suspect_targets = pd.concat(
        [
            targets_from_shared_case_locations,
            targets_from_existing_suspect_roles,
            targets_from_related_to_victims,
            targets_from_person_case_context,
        ],
        ignore_index=True
    )

    likely_suspect_targets = (
        likely_suspect_targets
        .dropna()
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates(subset=["person", "case_id"])
    )

    likely_suspect_targets.to_csv(
        "likely_suspect_targets.tsv",
        sep="\t",
        header=False,
        index=False
    )

    model.get_predicate("LikelySuspectInCase").add_data_file(
        Partition.TARGETS,
        "likely_suspect_targets.tsv"
    )

def run_psl():
    model = build_psl_model()
    load_model(model)
    fill_prediate_into_model(model)
    return model.infer()

def get_result_by_predicate_name(results, target_name):
    for predicate, df in results.items():
        if predicate.name().lower() == target_name.lower():
            return df

    raise KeyError(
        f"{target_name} not found. Available: {[p.name() for p in results.keys()]}"
    )

In [ ]:
results = run_psl()

In [ ]:
nodes_df = run_cypher("""
MATCH (n)
RETURN DISTINCT
  coalesce(n.id, n.name, toString(elementId(n))) AS id,
  labels(n)[0] AS label,
  coalesce(n.name, n.id, toString(elementId(n))) AS title
""")

edges_df = run_cypher("""
MATCH (a)-[r]->(b)
RETURN
  coalesce(a.id, a.name, toString(elementId(a))) AS source,
  coalesce(b.id, b.name, toString(elementId(b))) AS target,
  type(r) AS relation,
  false AS inferred,
  null AS score
""")

observed_edges = edges_df[["source", "target", "relation", "inferred", "score"]]

likely_location_predicate = "LikelyLocationInCase"

location_preds = get_result_by_predicate_name(
    results,
    "LikelyLocationInCase"
).rename(columns={
    0: "person",
    1: "case_id",
    2: "location",
    "truth": "score"
})

location_preds = location_preds[location_preds["score"] >= 0.5].copy()

location_preds["source"] = location_preds["person"]
location_preds["target"] = location_preds["location"]
location_preds["relation"] = location_preds.apply(
    lambda row: f"LIKELY_LOCATION_IN_CASE:{row['case_id']}",
    axis=1
)
location_preds["inferred"] = True

location_inferred_edges = location_preds[
    ["source", "target", "relation", "inferred", "score"]
]

likely_suspect_predicate = "LikelySuspectInCase"

suspect_preds = get_result_by_predicate_name(
    results,
    "LikelySuspectInCase"
).rename(columns={
    0: "person",
    1: "case_id",
    "truth": "score"
})

suspect_preds = suspect_preds[suspect_preds["score"] >= 0].copy()

suspect_preds["source"] = suspect_preds["person"]
suspect_preds["target"] = suspect_preds["case_id"]
suspect_preds["relation"] = "LIKELY_SUSPECT_IN_CASE"
suspect_preds["inferred"] = True

suspect_inferred_edges = suspect_preds[
    ["source", "target", "relation", "inferred", "score"]
]

graph_view_edges = pd.concat(
    [
        observed_edges,
        location_inferred_edges,
        suspect_inferred_edges
    ],
    ignore_index=True
)

In [ ]:
graph_view_nodes = nodes_df.copy()

net = Network(
    height="750px",
    width="100%",
    directed=True,
    notebook=True
)

net.repulsion(
    node_distance=300,
    central_gravity=0.1,
    spring_length=250,
    spring_strength=0.05,
    damping=0.09
)

for _, row in graph_view_nodes.iterrows():
    net.add_node(
        row["id"],
        label=row["title"],
        title=f"{row['label']}: {row['title']}"
    )

for _, row in graph_view_edges.iterrows():
    if row["inferred"]:
        label = f"{row['relation']} ({row['score']:.2f})"
        color = "red"
        dashes = True
    else:
        label = row["relation"]
        color = "gray"
        dashes = False

    net.add_edge(
        row["source"],
        row["target"],
        label=label,
        title=label,
        color=color,
        dashes=dashes
    )

net.write_html("graph_with_psl_view.html")

In [92]:
location_preds

,person,case_id,location,score,source,target,relation,inferred
0,CB,CASE_MM,PRAIA_DA_LUZ,1.000000,CB,PRAIA_DA_LUZ,LIKELY_LOCATION_IN_CASE:CASE_MM,True
1,MM,CASE_MM,APT_5A,1.000000,MM,APT_5A,LIKELY_LOCATION_IN_CASE:CASE_MM,True
2,RF,CASE_KS,RF_HOME,1.000000,RF,RF_HOME,LIKELY_LOCATION_IN_CASE:CASE_KS,True
3,PF,CASE_KS,PF_HOME_LA,1.000000,PF,PF_HOME_LA,LIKELY_LOCATION_IN_CASE:CASE_KS,True
4,PF,CASE_KS,SANTA_LUCIA,1.000000,PF,SANTA_LUCIA,LIKELY_LOCATION_IN_CASE:CASE_KS,True
5,KS,CASE_KS,SANTA_LUCIA,1.000000,KS,SANTA_LUCIA,LIKELY_LOCATION_IN_CASE:CASE_KS,True
6,FS,CASE_MB,SPENCER_AREA,1.000000,FS,SPENCER_AREA,LIKELY_LOCATION_IN_CASE:CASE_MB,True
7,MAGI,CASE_MB,WARREN_HOME,1.000000,MAGI,WARREN_HOME,LIKELY_LOCATION_IN_CASE:CASE_MB,True
8,MB,CASE_MB,WARREN_HOME,1.000000,MB,WARREN_HOME,LIKELY_LOCATION_IN_CASE:CASE_MB,True
9,MB,CASE_MB,COMINS_POND,1.000000,MB,COMINS_POND,LIKELY_LOCATION_IN_CASE:CASE_MB,True


In [93]:
suspect_preds

,person,case_id,score,source,target,relation,inferred
0,CB,CASE_MM,1.000000e+00,CB,CASE_MM,LIKELY_SUSPECT_IN_CASE,True
1,RF,CASE_KS,1.000000e+00,RF,CASE_KS,LIKELY_SUSPECT_IN_CASE,True
2,PF,CASE_KS,1.000000e+00,PF,CASE_KS,LIKELY_SUSPECT_IN_CASE,True
3,FS,CASE_MB,1.000000e+00,FS,CASE_MB,LIKELY_SUSPECT_IN_CASE,True
4,SS,CASE_KS,1.000000e+00,SS,CASE_KS,LIKELY_SUSPECT_IN_CASE,True
5,DS,CASE_KS,1.000000e+00,DS,CASE_KS,LIKELY_SUSPECT_IN_CASE,True
6,TD,CASE_KS,2.748163e-01,TD,CASE_KS,LIKELY_SUSPECT_IN_CASE,True
7,CA,CASE_KS,8.307296e-02,CA,CASE_KS,LIKELY_SUSPECT_IN_CASE,True
8,HEATHER,CASE_MB,4.284181e-13,HEATHER,CASE_MB,LIKELY_SUSPECT_IN_CASE,True
9,MAGI,CASE_MB,5.876563e-13,MAGI,CASE_MB,LIKELY_SUSPECT_IN_CASE,True


# Hide And Recover Test

In [ ]:
def hide_test_edge(person_id: str, case_id: str, rel_type: str):
    q = f"""
    MATCH (p:Person {{id: $personId}})-[r:{rel_type}]->(c:Case {{id: $caseId}})
    SET r.hidden_for_test = true
    RETURN p.id AS personId, type(r) AS relType, c.id AS caseId
    """
    return run_cypher(q, {
        "personId": person_id,
        "caseId": case_id
    })

def hit_at_k(preds, person_id, case_id, k=10):
    top_k = preds.head(k)
    return int(
        ((top_k["personId"] == person_id) &
         (top_k["caseId"] == case_id)).any()
    )

def hit_location_at_k(preds, case_id, location_id, k=10):
    top_k = preds.head(k)
    return int(
        ((top_k["caseId"] == case_id) &
         (top_k["locationId"] == location_id)).any()
    )

def test_hidden_suspect_recovery(
    person_id: str,
    case_id: str,
    prediction_fn=predict_suspect_links_common_neighbors,
    k: int = 10,
    limit: int = 50
):
    """
    Hide a known suspect edge, run prediction,
    check if it is recovered in top-k,
    then restore the edge.
    """

    try:
        # Hide edge
        run_cypher("""
        MATCH (:Person {id:$personId})-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(:Case {id:$caseId})
        SET r.hidden_for_test = true
        """, {
            "personId": person_id,
            "caseId": case_id
        })

        preds = prediction_fn(
            limit=limit,
            case_id=case_id
        )

        recovered_rows = preds[
            (preds["personId"] == person_id) &
            (preds["caseId"] == case_id)
        ]

        rank = None
        if not recovered_rows.empty:
            rank = recovered_rows.index[0] + 1

        hit = hit_at_k(
            preds,
            person_id,
            case_id,
            k=k
        )

        return {
            "person_id": person_id,
            "case_id": case_id,
            "hit_at_k": hit,
            "rank": rank,
            "top_k": preds.head(k)
        }

    finally:
        run_cypher("""
        MATCH (:Person {id:$personId})-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(:Case {id:$caseId})
        REMOVE r.hidden_for_test
        """, {
            "personId": person_id,
            "caseId": case_id
        })

def test_hidden_location_recovery(
    case_id: str,
    location_id: str,
    prediction_fn=predict_case_location_common_neighbors,
    k: int = 10,
    limit: int = 50
):
    """
    Hide a known case-location edge, run prediction,
    check if it is recovered in top-k,
    then restore the edge.
    """

    try:
        run_cypher("""
        MATCH (:Case {id:$caseId})-[r:HAS_LOCATION|FOUND_AT|SEARCHED_AT|INVESTIGATED_IN|RESCUED_AT]->(:Location {id:$locationId})
        SET r.hidden_for_test = true
        RETURN count(r) AS hidden_edges
        """, {
            "caseId": case_id,
            "locationId": location_id
        })

        preds = prediction_fn(
            limit=limit,
            case_id=case_id
        )

        recovered_rows = preds[
            (preds["caseId"] == case_id) &
            (preds["locationId"] == location_id)
        ]

        rank = None
        if not recovered_rows.empty:
            rank = recovered_rows.index[0] + 1

        hit = hit_location_at_k(
            preds,
            case_id,
            location_id,
            k=k
        )

        return {
            "case_id": case_id,
            "location_id": location_id,
            "hit_at_k": hit,
            "rank": rank,
            "top_k": preds.head(k)
        }

    finally:
        run_cypher("""
        MATCH (:Case {id:$caseId})-[r:HAS_LOCATION|FOUND_AT|SEARCHED_AT|INVESTIGATED_IN|RESCUED_AT]->(:Location {id:$locationId})
        REMOVE r.hidden_for_test
        """, {
            "caseId": case_id,
            "locationId": location_id
        })

## Test Suspect Recovery

In [ ]:
suspect_test_candidates = run_cypher("""
    MATCH (p:Person)-[pc]->(c:Case)
    OPTIONAL MATCH (p)-[pl]->(l:Location)

    WHERE type(pc) IN [
        'SUSPECT_IN',
        'ACCUSED_IN',
        'PERSON_OF_INTEREST'
    ]

    AND (
        l IS NULL OR type(pl) IN [
            'FOUND_AT',
            'LAST_SEEN_AT',
            'LAST_SEEN_NEAR',
            'PRESENT_NEAR',
            'RESCUED_AT',
            'STAYED_AT',
            'LIVED_AT',
            'LIVED_IN_OR_WORKED_NEAR'
        ]
    )

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        collect(DISTINCT l.id) AS connectedLocations,
        collect(DISTINCT l.name) AS connectedLocationNames,
        collect(DISTINCT type(pc)) AS caseRelationships,
        collect(DISTINCT type(pl)) AS locationRelationships,
        count(DISTINCT l) AS locationEvidenceCount

    ORDER BY locationEvidenceCount DESC, caseId, personId
    LIMIT 50
    """)

suspect_tests = (
    suspect_test_candidates
    [["personId", "caseId"]]
    .drop_duplicates()
    .head(10)
)

suspect_tests = list(suspect_tests.itertuples(index=False, name=None))

prediction_methods = {
    "common_neighbors": predict_suspect_links_common_neighbors,
    "adamic_adar": predict_suspect_links_adamic_adar,
    "preferential_attachment": predict_suspect_links_preferential_attachment
}

results = []

for person_id, case_id in suspect_tests:
    for method_name, prediction_fn in prediction_methods.items():
        result = test_hidden_suspect_recovery(
            person_id=person_id,
            case_id=case_id,
            prediction_fn=prediction_fn,
            k=5,
            limit=50
        )

        results.append({
            "method": method_name,
            "person_id": result["person_id"],
            "case_id": result["case_id"],
            "hit_at_5": result["hit_at_k"],
            "rank": result["rank"]
        })

results_suspect_df = pd.DataFrame(results)

In [ ]:
location_test_candidates = run_cypher("""
    MATCH (p:Person)-[pc]->(c:Case)
    MATCH (p)-[pl]->(l:Location)

    WHERE type(pc) IN [
        'MISSING_PERSON_IN',
        'POTENTIAL_VICTIM_IN',
        'RESCUED_IN',
        'VICTIM_IN',
        'SUSPECT_IN',
        'ACCUSED_IN',
        'PERSON_OF_INTEREST'
    ]

    AND type(pl) IN [
        'FOUND_AT',
        'LAST_SEEN_AT',
        'LAST_SEEN_NEAR',
        'PRESENT_NEAR',
        'RESCUED_AT',
        'STAYED_AT',
        'LIVED_AT',
        'LIVED_IN_OR_WORKED_NEAR'
    ]

    RETURN
        c.id AS caseId,
        c.name AS caseName,
        l.id AS locationId,
        l.name AS locationName,
        collect(DISTINCT p.id) AS peopleConnectingCaseToLocation,
        collect(DISTINCT type(pc)) AS caseRelationships,
        collect(DISTINCT type(pl)) AS locationRelationships,
        count(DISTINCT p) AS sharedPeople

    ORDER BY sharedPeople DESC, caseId, locationId
    LIMIT 50
    """)

location_tests = (
    location_test_candidates
    [["caseId", "locationId"]]
    .drop_duplicates()
    .head(10)
)

location_tests = list(location_tests.itertuples(index=False, name=None))

prediction_methods = {
    "common_neighbors": predict_case_location_common_neighbors,
    "adamic_adar": predict_case_location_adamic_adar,
    "preferential_attachment": predict_case_location_preferential_attachment
}

location_results = []

for case_id, location_id in location_tests:
    for method_name, prediction_fn in prediction_methods.items():

        result = test_hidden_location_recovery(
            case_id=case_id,
            location_id=location_id,
            prediction_fn=prediction_fn,
            k=5,
            limit=50
        )

        location_results.append({
            "method": method_name,
            "case_id": result["case_id"],
            "location_id": result["location_id"],
            "hit_at_5": result["hit_at_k"],
            "rank": result["rank"]
        })

results_location_df = pd.DataFrame(location_results)

# Test PSL Logic Layer

In [ ]:
def test_psl_location_recovery(
    person_id: str,
    case_id: str,
    location_id: str,
    run_psl_fn,
    k: int = 10
):
    try:
        run_cypher("""
        MATCH (:Person {id:$personId})-[r:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT|LIVED_AT|LIVED_IN_OR_WORKED_NEAR]->(:Location {id:$locationId})
        SET r.hidden_for_test = true
        """, {
            "personId": person_id,
            "locationId": location_id
        })

        results = run_psl_fn()

        preds = get_result_by_predicate_name(
            results,
            "LikelyLocationInCase"
        ).rename(columns={
            0: "person",
            1: "case_id",
            2: "location",
            "truth": "score"
        })

        preds = preds.sort_values("score", ascending=False).reset_index(drop=True)

        match = preds[
            (preds["person"] == person_id) &
            (preds["case_id"] == case_id) &
            (preds["location"] == location_id)
        ]

        rank = None if match.empty else match.index[0] + 1

        return {
            "person_id": person_id,
            "case_id": case_id,
            "location_id": location_id,
            "hit_at_k": int(rank is not None and rank <= k),
            "rank": rank,
            "score": None if match.empty else match.iloc[0]["score"],
            "top_k": preds.head(k)
        }

    finally:
        run_cypher("""
        MATCH (:Person {id:$personId})-[r:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT|LIVED_AT|LIVED_IN_OR_WORKED_NEAR]->(:Location {id:$locationId})
        REMOVE r.hidden_for_test
        """, {
            "personId": person_id,
            "locationId": location_id
        })

In [ ]:
psl_location_test_candidates = run_cypher("""
    MATCH (p:Person)-[pc]->(c:Case)
    MATCH (p)-[pl]->(l:Location)

    WHERE type(pc) IN [
        'MISSING_PERSON_IN',
        'POTENTIAL_VICTIM_IN',
        'RESCUED_IN',
        'VICTIM_IN',
        'SUSPECT_IN',
        'ACCUSED_IN',
        'PERSON_OF_INTEREST'
    ]

    AND type(pl) IN [
        'FOUND_AT',
        'LAST_SEEN_AT',
        'LAST_SEEN_NEAR',
        'PRESENT_NEAR',
        'RESCUED_AT',
        'STAYED_AT',
        'LIVED_AT',
        'LIVED_IN_OR_WORKED_NEAR'
    ]

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        l.id AS locationId,
        l.name AS locationName,
        type(pc) AS caseRelationship,
        type(pl) AS locationRelationship

    ORDER BY caseId, personId, locationId
    LIMIT 50
    """)

psl_location_tests = (
    psl_location_test_candidates
    [["personId", "caseId", "locationId"]]
    .drop_duplicates()
    .head(10)
)

psl_location_tests = list(psl_location_tests.itertuples(index=False, name=None))

location_results = [
    test_psl_location_recovery(p, c, l, run_psl, k=5)
    for p, c, l in  psl_location_tests
]

psl_results_location_df = pd.DataFrame(location_results)

In [ ]:
def test_psl_suspect_recovery(
    person_id: str,
    case_id: str,
    run_psl_fn,
    k: int = 5
):
    try:
        run_cypher("""
        MATCH (:Person {id:$personId})-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(:Case {id:$caseId})
        SET r.hidden_for_test = true
        """, {"personId": person_id, "caseId": case_id})

        results = run_psl_fn()

        preds = get_result_by_predicate_name(
            results,
            "LikelySuspectInCase"
        ).rename(columns={
            0: "person",
            1: "case_id",
            "truth": "score"
        })

        preds = preds.sort_values("score", ascending=False).reset_index(drop=True)

        match = preds[
            (preds["person"] == person_id) &
            (preds["case_id"] == case_id)
        ]

        rank = None if match.empty else match.index[0] + 1

        return {
            "person_id": person_id,
            "case_id": case_id,
            "hit_at_k": int(rank is not None and rank <= k),
            "rank": rank,
            "score": None if match.empty else match.iloc[0]["score"],
            "top_k": preds.head(k)
        }

    finally:
        run_cypher("""
        MATCH (:Person {id:$personId})-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(:Case {id:$caseId})
        REMOVE r.hidden_for_test
        """, {"personId": person_id, "caseId": case_id})

In [ ]:
suspect_results = [
    test_psl_suspect_recovery(p, c, run_psl, k=5)
    for p, c in suspect_tests
]

psl_results_suspect_df = pd.DataFrame(suspect_results)

# Test Results

In [ ]:
def summarize_psl_results(results_df, k=5):
    df = results_df.copy()

    # reciprocal rank
    df["reciprocal_rank"] = (
        1 / df["rank"]
    ).replace([float("inf")], 0)

    df["reciprocal_rank"] = df["reciprocal_rank"].fillna(0)

    summary = pd.DataFrame({
        "method": ["psl"],
        f"hit_rate_at_{k}": [df["hit_at_k"].mean()],
        "coverage": [df["rank"].notna().mean()],
        "average_rank": [df["rank"].mean()],
        "median_rank": [df["rank"].median()],
        "mrr": [df["reciprocal_rank"].mean()],
        "tests": [len(df)],
        "recovered": [df["rank"].notna().sum()]
    })

    return summary

def summarize_embedding_results(df, hit_col="hit_at_5"):
    temp = df.copy()
    temp["reciprocal_rank"] = (1 / temp["rank"]).fillna(0)

    hit_rate_col = f"hit_rate_{hit_col.removeprefix('hit_')}"

    return (
        temp
        .groupby("method")
        .agg(**{
            hit_rate_col: (hit_col, "mean"),
            "coverage": ("rank", lambda x: x.notna().mean()),
            "average_rank": ("rank", "mean"),
            "median_rank": ("rank", "median"),
            "mrr": ("reciprocal_rank", "mean"),
            "tests": (hit_col, "count"),
            "recovered": ("rank", lambda x: x.notna().sum())
        })
        .reset_index()
    )

summary_df = summarize_embedding_results(
    results_suspect_df,
    hit_col="hit_at_5"
)

location_summary_df = summarize_embedding_results(
    results_location_df,
    hit_col="hit_at_5"
)

psl_suspect_summary = summarize_psl_results(
    psl_results_suspect_df,
    k=5
)

psl_location_summary = summarize_psl_results(
    psl_results_location_df,
    k=5
)



In [94]:
summary_df

,method,hit_rate_at_5,coverage,average_rank,median_rank,mrr,tests,recovered
0,adamic_adar,0.8,0.8,3.250000,3.5,0.356667,10,8
1,common_neighbors,0.8,0.8,3.125000,3.5,0.328333,10,8
2,preferential_attachment,0.1,0.9,13.666667,13.0,0.160533,10,9


In [95]:
location_summary_df

,method,hit_rate_at_5,coverage,average_rank,median_rank,mrr,tests,recovered
0,adamic_adar,1.00,1.0,2.000,2.0,0.635417,8,8
1,common_neighbors,1.00,1.0,1.875,2.0,0.656250,8,8
2,preferential_attachment,0.25,1.0,7.875,7.5,0.331969,8,8


In [96]:
psl_suspect_summary

,method,hit_rate_at_5,coverage,average_rank,median_rank,mrr,tests,recovered
0,psl,0.5,0.8,5.375,4.5,0.263182,10,8


In [97]:
psl_location_summary

,method,hit_rate_at_5,coverage,average_rank,median_rank,mrr,tests,recovered
0,psl,0.3,0.7,6.0,6.0,0.220043,10,7
